In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/motion-s-hierarchical-text-to-motion-generation-for-sign-language/sample_submission.csv
/kaggle/input/motion-s-hierarchical-text-to-motion-generation-for-sign-language/train.csv
/kaggle/input/motion-s-hierarchical-text-to-motion-generation-for-sign-language/test.csv
/kaggle/input/motion-s-hierarchical-text-to-motion-generation-for-sign-language/Motion-Features/2648488.npy
/kaggle/input/motion-s-hierarchical-text-to-motion-generation-for-sign-language/Motion-Features/1647925.npy
/kaggle/input/motion-s-hierarchical-text-to-motion-generation-for-sign-language/Motion-Features/5146859.npy
/kaggle/input/motion-s-hierarchical-text-to-motion-generation-for-sign-language/Motion-Features/3912557.npy
/kaggle/input/motion-s-hierarchical-text-to-motion-generation-for-sign-language/Motion-Features/2641776.npy
/kaggle/input/motion-s-hierarchical-text-to-motion-generation-for-sign-language/Motion-Features/2735152.npy
/kaggle/input/motion-s-hierarchical-text-to-motion-generation-for-sign-

In [2]:
import pandas as pd
import numpy as np
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- 1. ROBUST DATA LOADING ---
# This function searches for the files in the most common Kaggle directories
def load_data(filename):
    potential_paths = [
        './',                                                                            # Current directory
        '/kaggle/input/motion-s-hierarchical-text-to-motion-generation-for-sign-language/', # Kaggle Default
        '/kaggle/input/motion-s/',                                                       # Alternate Kaggle
        '../input/motion-s-hierarchical-text-to-motion-generation-for-sign-language/'   # Jupyter parent
    ]
    
    for path in potential_paths:
        full_path = os.path.join(path, filename)
        if os.path.exists(full_path):
            print(f"Found {filename} at: {full_path}")
            return pd.read_csv(full_path)
    
    # If code reaches here, the file wasn't found
    print(f"\nCRITICAL ERROR: Could not find {filename}.")
    print(f"Current Working Directory: {os.getcwd()}")
    print("Files in Current Directory:", os.listdir(os.getcwd()))
    raise FileNotFoundError(f"Please make sure the dataset is attached to the notebook.")

# Load the datasets
try:
    print("--- Loading Data ---")
    train_df = load_data('train.csv')
    test_df = load_data('test.csv')
except FileNotFoundError as e:
    print(e)
    # Stop execution if files are missing
    raise

# --- 2. PREPROCESSING & FILTERING ---
print("\n--- Filtering Training Data ---")
# Drop rows with missing values in critical columns
train_df = train_df.dropna(subset=['gloss', 'base_tokens', 'residual_1', 'residual_2', 'residual_3', 'residual_4', 'residual_5'])

# Helper function to check token length (must be 40-800)
def get_token_len(s):
    try:
        return len(str(s).split())
    except:
        return 0

train_df['token_len'] = train_df['base_tokens'].apply(get_token_len)

# Keep only valid training samples
valid_train = train_df[(train_df['token_len'] >= 40) & (train_df['token_len'] <= 800)].copy().reset_index(drop=True)
print(f"Original training samples: {len(train_df)}")
print(f"Valid training samples used: {len(valid_train)}")

# --- 3. RETRIEVAL (TF-IDF) ---
print("\n--- Vectorizing Glosses ---")
# Combine train and test glosses to fit the vectorizer (ensures vocabulary covers both)
all_glosses = pd.concat([valid_train['gloss'], test_df['gloss']]).fillna("")

# Use word-level n-grams to capture phrases (e.g., "GO STORE")
vectorizer = TfidfVectorizer(analyzer='word', token_pattern=r'\b\w+\b', ngram_range=(1, 2))
vectorizer.fit(all_glosses)

train_tfidf = vectorizer.transform(valid_train['gloss'].fillna(""))
test_tfidf = vectorizer.transform(test_df['gloss'].fillna(""))

# --- 4. SIMILARITY SEARCH ---
print("\n--- Matching Test Samples to Training Data ---")
# Compute cosine similarity between every Test sample and every Train sample
similarities = cosine_similarity(test_tfidf, train_tfidf)

# Find the index of the best match for each test sample
best_match_indices = np.argmax(similarities, axis=1)

# --- 5. CREATE SUBMISSION ---
print("\n--- constructing submission.csv ---")
submission = pd.DataFrame()
submission['id'] = test_df['id']

# Retrieve tokens from the matched training rows
matched_rows = valid_train.iloc[best_match_indices]
token_cols = ['base_tokens', 'residual_1', 'residual_2', 'residual_3', 'residual_4', 'residual_5']

for col in token_cols:
    submission[col] = matched_rows[col].values

# --- 6. SAVE OUTPUT ---
output_file = 'submission.csv'
submission.to_csv(output_file, index=False)
print(f"Success! {output_file} generated with {len(submission)} rows.")
print("\nFirst 5 rows of submission:")
print(submission.head())

--- Loading Data ---
Found train.csv at: /kaggle/input/motion-s-hierarchical-text-to-motion-generation-for-sign-language/train.csv
Found test.csv at: /kaggle/input/motion-s-hierarchical-text-to-motion-generation-for-sign-language/test.csv

--- Filtering Training Data ---
Original training samples: 12463
Valid training samples used: 12373

--- Vectorizing Glosses ---

--- Matching Test Samples to Training Data ---

--- constructing submission.csv ---
Success! submission.csv generated with 3000 rows.

First 5 rows of submission:
        id                                        base_tokens  \
0  6420249  379 295 139 139 139 139 139 61 139 139 139 139...   
1  6420682  130 260 9 207 207 207 449 277 457 212 212 268 ...   
2  6425789  379 295 139 139 139 139 139 61 139 139 139 139...   
3  6425858  130 276 174 174 145 145 145 174 174 50 50 50 2...   
4  6427530  130 326 95 95 95 95 95 95 95 95 95 95 95 95 95...   

                                          residual_1  \
0  314 441 441 333 8